# Register adjoints

An adjoint is the conjugate transpose of a linear operator. This complete native mapping follows the 17 cells of the frozen reference tutorial. Preparation becomes an effect, and a composed operation reverses its calls and replaces each leaf with its exact adjoint. Diagrams flow from left to right.

Adapted from the Apache-2.0 licensed 2023 Google LLC tutorial at locked commit bcb17bb25360ed68ce3bc084645db3e929799389. Native ownership and interface details are recorded in REGISTER_ADJOINT_TUTORIAL_AUDIT.md.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

import torch
from graphviz import Source
from IPython.display import display

from examples.algorithms.adjoint_programs import (
    bell_preparation,
    phase_program,
    serial_atoms,
    tutorial_adjoint,
    tutorial_definitions,
)
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import (
    ConjunctionCall,
)
from flagquantum.compiler.registers.component_tensor_expansion import (
    expand_component_root_for_tensor,
)
from flagquantum.compiler.registers.primitives.and_compute import (
    and_compute_ir,
    and_uncompute_ir,
)
from flagquantum.compiler.registers.primitives.power import register_power_ir
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir import CircuitIR, Instruction
from flagquantum.core.ir.program import ProgramRecord
from flagquantum.drawer.component_graph import export_component_graph_dot
from flagquantum.simulation.unitary import get_unitary


def show_program(record: ProgramRecord, entry: str = "main") -> Source:
    graph = Source(export_component_graph_dot(record, entry))
    cast(Callable[[object], None], display)(graph)
    return graph


definitions = tutorial_definitions()
zero_program, _ = expand_component_root_for_tensor(
    ProgramRecord(definitions),
    lambda name: True,
    entry="zero",
    full_flatten=False,
)
zero_effect_program, _ = expand_component_root_for_tensor(
    ProgramRecord(definitions),
    lambda name: True,
    entry="zero_effect",
    full_flatten=False,
)
zero_graph = show_program(zero_program, "tensor_root")
zero_effect_graph = show_program(zero_effect_program, "tensor_root")


## Interface

A Program stores calls and typed ports. `tutorial_adjoint` uses the shared Compiler hierarchy transformation with explicit exact leaf definitions. For two operations the reversed program implements the identity $(AB)^\dagger=B^\dagger A^\dagger$. Four newly prepared bits belong to the returned output words.


In [ ]:
bell = bell_preparation()
bell_graph = show_program(bell)
bell_bound = bind_register_program(
    bell, (), clean=(0, 1, 2, 3), allow_workspace_outputs=True
)
bell_circuit = Circuit(4, dtype=torch.complex128).append(
    FieldOperation(4, bell_bound.circuit.instructions)
)
bell_vector = bell_circuit.state().flatten()
assert bell_bound.outputs == ((0,), (1,), (2,), (3,))
assert torch.count_nonzero(bell_vector[1:-1]) == 0
assert torch.allclose(
    bell_vector[[0, 15]], torch.ones(2, dtype=torch.complex128) / 2**0.5
)


In [ ]:
bell_adjoint = tutorial_adjoint(bell)
bell_adjoint_graph = show_program(bell_adjoint)
bell_inverse_bound = bind_register_program(
    bell_adjoint,
    ((0,), (1,), (2,), (3,)),
    input_ownership=((0, False), (1, False), (2, False), (3, False)),
)
recovered = (
    Circuit(4, inputs=bell_vector, dtype=torch.complex128)
    .append(FieldOperation(4, bell_inverse_bound.circuit.instructions))
    .state()
    .flatten()
)
assert torch.count_nonzero(recovered[1:]) == 0
assert torch.allclose(recovered[0], torch.ones((), dtype=torch.complex128))


## Specialized exact leaves

Self-adjoint X and CNOT retain their primitive instruction. Measurement-assisted And erasure has its own implementation. A gate-by-gate inverse of the compute network would have a different resource cost.


In [ ]:
self_adjoint = {}
for name, wires in (("x", (0,)), ("cx", (0, 1))):
    body = CircuitIR(len(wires), (Instruction(name, wires),))
    self_adjoint[name] = register_power_ir(body, 1, adjoint=True) == body.instructions
assert all(self_adjoint.values())
print(self_adjoint)


In [ ]:
and_forward = and_compute_ir(0, 1, 2)
and_backward = and_uncompute_ir(0, 1, 2, 0)
emitted_counts = {
    "compute": Counter(op.name for op in and_forward),
    "erase": Counter(op.name for op in and_backward),
}
modeled_costs = {
    "compute": query_qec_gate_cost(ConjunctionCall((1, 1))),
    "erase": query_qec_gate_cost(ConjunctionCall((1, 1), adjoint=True)),
}
assert len(and_forward) == 13 and len(and_backward) == 4
print("Emitted primitive instructions", emitted_counts)
print("Independent QEC category models", modeled_costs)


## Stored-body fallback

The Compiler derives the inverse of a stored composite from its child calls. Atomic leaves require explicitly supplied exact adjoints. This keeps measurement and ownership semantics with their existing primitive owners.


In [ ]:
serial = serial_atoms()
serial_adjoint = tutorial_adjoint(serial)
serial_adjoint


The native equivalents are split by responsibility.

- Core Register and Signature reverse LEFT and RIGHT ports.
- Compiler reverses stored calls, preserves shared definitions and selects specialized exact leaves.
- Resource accounting counts the transformed graph independently of emitted instructions.
- Ecosystem exposes a bound body through its SDK adapter.
- Drawer reverses directional glyphs and toggles dagger text through `adjoint_wire_symbol`.
- Names remain caller-owned native definition names.

Classical actions and tensor suppliers remain explicit numerical inputs. A general Python function is not inverted. The complete reference wrapper's automatic diagram-exponent delegation and wider metadata protocols remain tracked separately in the full-file audit.


In [ ]:
serial_graph = show_program(serial)
serial_adjoint_graph = show_program(serial_adjoint)


The next program applies H followed by T. Its inverse applies T-dagger followed by H, so complete complex amplitudes verify the relative and overall phase.


In [ ]:
phase = phase_program()
phase_graph = show_program(phase)


In [ ]:
phase_adjoint = tutorial_adjoint(phase)
phase_adjoint_graph = show_program(phase_adjoint)


In [ ]:
phase_bound = bind_register_program(phase, ((0,),))
phase_inverse_bound = bind_register_program(phase_adjoint, ((0,),))
unitary = get_unitary(phase_bound.circuit, dtype=torch.complex128)
adjoint_unitary = get_unitary(phase_inverse_bound.circuit, dtype=torch.complex128)
torch.testing.assert_close(unitary.conj().T, adjoint_unitary, atol=3e-16, rtol=0)
phase_counts = Counter(op.name for op in phase_bound.circuit.instructions)
phase_adjoint_counts = Counter(
    op.name for op in phase_inverse_bound.circuit.instructions
)
print("Check!", phase_counts, phase_adjoint_counts)


## Additional functionality

Register and Signature have `adjoint()` methods. Drawer `adjoint_wire_symbol` returns an independent glyph with reversed arrows or toggled dagger text. The zero effect is a projection in tensor inspection. Physical release requires the clean-work promise and is verified below only on the prepared GHZ input. Primitive counts and the independently declared QEC model remain separate.
